## Multiuser support for the coversational chat bot

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.documents import Document

from langchain.chains import create_history_aware_retriever,create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda, RunnableParallel

from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

from dotenv import load_dotenv
from typing import List
from pydantic import BaseModel, Field
from datetime import datetime
import sqlite3
import uuid
import os

In [2]:
# loading environment variables for API keys
load_dotenv()

# Update LangSmith project name for this notebook
os.environ["LANGCHAIN_PROJECT"] = "Multiuser Conversational Chatbot"

# creating llm and embeddings
llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0
)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [3]:
def load_multiple_documents(directory_path: str) -> List[Document]:

    documents = []

    for file in os.listdir(directory_path):
        if file.endswith(".docx"):
            loader = Docx2txtLoader(os.path.join(directory_path, file))
        elif file.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(directory_path, file))
        else:
            print(f"Skipping {file} as it is not a supported file type")
            continue
        documents.extend(loader.load())

    return documents

def docs_to_text(documents: List[Document]) -> str:
    return "\n\n".join([doc.page_content for doc in documents])

In [4]:
# loading documents
document_path = "docs"
documents = load_multiple_documents(document_path)
print(f"Number of documents: {len(documents)}")

# splitting documents into consumable chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100, length_function=len)
splits = text_splitter.split_documents(documents)
print(f"Number of documents after splitting: {len(splits)}")

# creating embeddings using OpenAI embeddings
doc_embeddings = embeddings.embed_documents([split.page_content for split in splits])
print(f"Embeddings created for {len(doc_embeddings)} documents")

# creating Chroma vector store and storing embeddings in it
collection_name = "docs_collection"
vectorstore = Chroma.from_documents(splits, embeddings, collection_name=collection_name, persist_directory="multiuser_chroma_db")
print("Document embeddings added to ChromaDB present in ./multiuser_chroma_db directory")

# creating retriever to get similar documents from vector store
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("Retriever created, to capture top 2 similar documents from vector stores")

# Setting up output parser - String output parser
parser = StrOutputParser()
print("Output parser created - String output parser for human readable output")

Number of documents: 5
Number of documents after splitting: 8
Embeddings created for 8 documents
Document embeddings added to ChromaDB present in ./multiuser_chroma_db directory
Retriever created, to capture top 2 similar documents from vector stores
Output parser created - String output parser for human readable output


In [5]:
# Setting up prompt template
template = """
Answer the question as truthfully as possible using the provided context,
and if the answer is not contained within the context, say "I don't know".

Context: {context}
Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)
print("Prompt template created")

Prompt template created


In [6]:
# creating RAG chain
rag_parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(docs_to_text),
    "question": RunnablePassthrough()
})

chain = rag_parallel_chain | prompt | llm | parser

print("RAG chain created")

RAG chain created


In [7]:
chat_history = []

query = "When was GreenGrow Innovations founded?"

chat_history.append(HumanMessage(content=query))
result = chain.invoke(query)
chat_history.append(AIMessage(content=result))

print(result)

print("Chat history: ", chat_history)

GreenGrow Innovations was founded in 2010.
Chat history:  [HumanMessage(content='When was GreenGrow Innovations founded?', additional_kwargs={}, response_metadata={}), AIMessage(content='GreenGrow Innovations was founded in 2010.', additional_kwargs={}, response_metadata={})]


In [8]:
contextualized_system_prompt = """
    Given a chat history and the latest user question which may reference context in the chat history,
    reformulate the question to be a standalone question that can be understood without the chat history.

    Do NOT answer the question, just reformulate it or return it as is.
    """
contextualized_question_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualized_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

print("Contextualized prompt template created, to capture context from chat history")

Contextualized prompt template created, to capture context from chat history


In [9]:
# this retriever will capture the contect and use it contextualized answer to retrieve similar documents
history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualized_question_prompt)

In [10]:
# prompt for question-answering - using chat history and context with stuff method - with user input
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that answers questions based on the provided context."),
    ("system", "Context: {context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# creating RAG chain using 
# history aware retriever (extracts relavent documents from vector store) - this provides context
# the context is now passed to question-answer chain 
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

# Now we can use the RAG chain to answer the next user query and we also need to pass the chat history here
query = "Where is it headquartered?"
rag_chain.invoke({"input": query, "chat_history": chat_history})

{'input': 'Where is it headquartered?',
 'chat_history': [HumanMessage(content='When was GreenGrow Innovations founded?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='GreenGrow Innovations was founded in 2010.', additional_kwargs={}, response_metadata={})],
 'context': [Document(id='b05ce26e-e4e4-4e31-824c-366122060dda', metadata={'source': 'docs/GreenGrow Innovations_ Company History.docx'}, page_content="The company's breakthrough came in 2018 with the introduction of the EcoHarvest System, an integrated solution that combined smart irrigation, soil monitoring, and automated harvesting techniques. This system caught the attention of large-scale farmers across the United States, propelling GreenGrow to national prominence.\n\n\n\nToday, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in v

### Now lets build multi user chatbot

In [14]:
DB_NAME = "multiuser_rag_app.db"

def get_db_connection():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    return conn

def create_application_logs_table():
    conn = get_db_connection()
    conn.execute("""
        CREATE TABLE IF NOT EXISTS application_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            session_id TEXT NOT NULL,
            user_query TEXT NOT NULL,
            llm_response TEXT NOT NULL,
            model_used TEXT NOT NULL,
            timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )"""
    )
    conn.commit()
    conn.close()

def insert_application_log(session_id, user_query, llm_response, model_used):
    conn = get_db_connection()
    conn.execute("INSERT INTO application_logs (session_id, user_query, llm_response, model_used) VALUES (?, ?, ?, ?)", 
                 (session_id, user_query, llm_response, model_used))
    conn.commit()
    conn.close()

def get_chat_history(session_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT user_query, llm_response FROM application_logs WHERE session_id = ? ORDER BY timestamp ASC", (session_id,))

    messages = []
    for row in cursor.fetchall():
        messages.append({"role": "user", "content": row[0]})
        messages.append({"role": "assistant", "content": row[1]})

    conn.close()
    return messages

# Create the application logs table
create_application_logs_table()

In [16]:
session_id = str(uuid.uuid4())
chat_history = get_chat_history(session_id)

print("Chat history: ", chat_history)

query_1 = "When was GreenGrow Innovations founded?"

result_1 = rag_chain.invoke({"input": query_1, "chat_history": chat_history})["answer"]
insert_application_log(session_id, query_1, result_1, "gpt-4o-mini")

print(f"Human Query: {query_1}")
print(f"AI Response: {result_1}")

Chat history:  []
Human Query: When was GreenGrow Innovations founded?
AI Response: GreenGrow Innovations was founded in 2010.


In [17]:
chat_history = get_chat_history(session_id)

print("Chat history: ", chat_history)

query_2 = "Where is it headquartered?"

result_2 = rag_chain.invoke({"input": query_2, "chat_history": chat_history})["answer"]
insert_application_log(session_id, query_2, result_2, "gpt-4o-mini")

print(f"Human Query: {query_2}")
print(f"AI Response: {result_2}")

Chat history:  [{'role': 'user', 'content': 'When was GreenGrow Innovations founded?'}, {'role': 'assistant', 'content': 'GreenGrow Innovations was founded in 2010.'}]
Human Query: Where is it headquartered?
AI Response: GreenGrow Innovations is headquartered in Portland, Oregon.


In [ ]:
chat_history = get_chat_history(session_id)
print("Chat history: ", chat_history)

Chat history:  [{'role': 'user', 'content': 'When was GreenGrow Innovations founded?'}, {'role': 'assistant', 'content': 'GreenGrow Innovations was founded in 2010.'}, {'role': 'user', 'content': 'Where is it headquartered?'}, {'role': 'assistant', 'content': 'GreenGrow Innovations is headquartered in Portland, Oregon.'}]
